# 03 – Preprocessing: Codificación de Variables Categóricas

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Transformar variables categóricas en representaciones numéricas compatibles con algoritmos de Machine Learning.

In [ ]:
import pandas as pd
import numpy as np
import os

# ─── Cargar datos ─────────────────────────────────────────────────────────────
PROC_PATH = os.path.join('..', 'data', 'processed', 'epen_filtered.csv')
try:
    df = pd.read_csv(PROC_PATH)
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'edad': np.random.randint(14, 70, n),
        'sexo': np.random.choice(['Hombre', 'Mujer'], n),
        'nivel_educativo': np.random.choice(
            ['Sin instrucción', 'Primaria', 'Secundaria', 'Preparatoria', 'Universidad', 'Posgrado'], n
        ),
        'estado_civil': np.random.choice(['Soltero', 'Casado', 'Unión libre', 'Divorciado', 'Viudo'], n),
        'ingreso_mensual': np.random.exponential(8000, n).round(2),
        'horas_trabajadas': np.random.randint(0, 60, n),
        'tipo_empleo': np.random.choice(['Formal', 'Informal', 'Sin empleo'], n),
        'sector': np.random.choice(['Agricultura', 'Industria', 'Comercio', 'Servicios', 'Gobierno'], n),
        'target_desocupado': np.random.choice([0, 1], n, p=[0.90, 0.10]),
    })

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print('Variables categóricas:', df.select_dtypes('object').columns.tolist())

## 1. Codificación ordinal – `nivel_educativo`

Esta variable tiene un orden natural, por lo que se mapea a valores enteros.

In [ ]:
orden_educacion = {
    'Sin instrucción': 0,
    'Primaria': 1,
    'Secundaria': 2,
    'Preparatoria': 3,
    'Universidad': 4,
    'Posgrado': 5,
}
df_enc = df.copy()
df_enc['nivel_educativo_ord'] = df_enc['nivel_educativo'].map(orden_educacion)
print(df_enc[['nivel_educativo', 'nivel_educativo_ord']].drop_duplicates().sort_values('nivel_educativo_ord'))

## 2. Codificación binaria – `sexo`, `tipo_empleo`

In [ ]:
# sexo: Hombre=1, Mujer=0
df_enc['sexo_bin'] = (df_enc['sexo'] == 'Hombre').astype(int)

# tipo_empleo: Formal=1, Informal=0, Sin empleo=-1
tipo_map = {'Formal': 1, 'Informal': 0, 'Sin empleo': -1}
df_enc['tipo_empleo_cod'] = df_enc['tipo_empleo'].map(tipo_map)

print(df_enc[['sexo', 'sexo_bin']].drop_duplicates())
print(df_enc[['tipo_empleo', 'tipo_empleo_cod']].drop_duplicates().sort_values('tipo_empleo_cod'))

## 3. One-Hot Encoding – `estado_civil`, `sector`

Variables nominales sin orden: se aplica One-Hot Encoding (OHE) con `drop_first=True` para evitar multicolinealidad.

In [ ]:
ohe_cols = ['estado_civil', 'sector']
df_enc = pd.get_dummies(df_enc, columns=ohe_cols, drop_first=True, dtype=int)

# Eliminar columnas originales ya codificadas
drop_orig = ['sexo', 'nivel_educativo', 'tipo_empleo']
df_enc = df_enc.drop(columns=[c for c in drop_orig if c in df_enc.columns])

print(f'Dataset codificado: {df_enc.shape[0]:,} filas × {df_enc.shape[1]} columnas')
df_enc.head()

In [ ]:
os.makedirs(os.path.join('..', 'data', 'processed'), exist_ok=True)
df_enc.to_csv(os.path.join('..', 'data', 'processed', 'epen_encoded.csv'), index=False)
print('Dataset guardado: epen_encoded.csv')